In [1]:
import pandas as pd
from statsmodels.stats.multitest import multipletests
import numpy as np
from pathlib import Path

def adjust_pvalue(group: str, base_dir: str = ".") -> pd.DataFrame:    
    base_path = Path(base_dir) / group
    pval_path = base_path / f"{group}_pvalues.tsv"

    # load p-values
    pval_df = pd.read_csv(pval_path, sep="\t", index_col=0)

    # Exclude diagonal and vectorize p-values
    mask = ~np.eye(len(pval_df), dtype=bool).flatten()
    pvals = pval_df.values.flatten()[mask]

    # Benjamini–Hochberg correction
    reject, pvals_corrected, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

    # Put corrected p-values back into matrix
    adj_pval_matrix = pval_df.copy()
    adj_pval_matrix.values[mask.reshape(pval_df.shape)] = pvals_corrected

    # Save (keep index)
    output_path = base_path / f"{group}_adj_pvalues.tsv"
    adj_pval_matrix.to_csv(output_path, sep='\t')

    return adj_pval_matrix

groups = ['all', 'c0', 'c1', 'c2', 'c3']
base_dir = "data/3-results/fastspar"  

for g in groups:
    adj_df = adjust_pvalue(g, base_dir=base_dir)